In [7]:
!pip install pyspark

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Netflix Data Pipeline") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .getOrCreate()

print("Spark with AWS support is running ✅")

Spark with AWS support is running ✅


In [9]:
from google.colab import files
uploaded = files.upload()

Saving netflix_titles.csv to netflix_titles (20).csv


In [10]:
df = spark.read.csv(
    "netflix_titles.csv",
    header=True,
    inferSchema=False,   # IMPORTANT CHANGE
    multiLine=True,      # handles broken rows
    escape='"'           # handles quotes inside text
)

In [11]:
from pyspark.sql.functions import col, trim

for column in df.columns:
    df = df.withColumn(column, trim(col(column)))

In [12]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast_members: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [13]:
df = df.dropDuplicates()

In [14]:
from pyspark.sql.functions import col, sum
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-------+----+-----+--------+------------+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast_members|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+------------+-------+----------+------------+------+--------+---------+-----------+
|      0|   0|    0|    2634|         825|    831|        10|           0|     4|       3|        0|          0|
+-------+----+-----+--------+------------+-------+----------+------------+------+--------+---------+-----------+



In [15]:
df = df.fillna({
    "director": "Unknown",
    "country": "Unknown",
    "rating": "Not Rated"
})

In [16]:
df = df.dropna(subset=["date_added"])

In [17]:
from pyspark.sql.functions import expr

df = df.withColumn(
    "date_added",
    expr("try_to_timestamp(date_added, 'MMMM d, yyyy')")
)

In [18]:
df = df.filter(col("date_added").isNotNull())

In [19]:
from pyspark.sql.functions import year

df = df.withColumn("year_added", year(col("date_added")))

In [20]:
from pyspark.sql.functions import when

df = df.withColumn(
    "content_category",
    when(col("type") == "Movie", "Film").otherwise("Series")
)

In [21]:
df.groupBy("type").count().show()

+-------+-----+
|   type|count|
+-------+-----+
|TV Show| 2666|
|  Movie| 6131|
+-------+-----+



In [22]:
df.groupBy("country").count().orderBy("count", ascending=False).show(10)

+--------------+-----+
|       country|count|
+--------------+-----+
| United States| 2812|
|         India|  972|
|       Unknown|  830|
|United Kingdom|  418|
|         Japan|  244|
|   South Korea|  199|
|        Canada|  181|
|         Spain|  145|
|        France|  124|
|        Mexico|  110|
+--------------+-----+
only showing top 10 rows


In [23]:
df.groupBy("year_added").count().orderBy("year_added").show()

+----------+-----+
|year_added|count|
+----------+-----+
|      2008|    2|
|      2009|    2|
|      2010|    1|
|      2011|   13|
|      2012|    3|
|      2013|   11|
|      2014|   24|
|      2015|   82|
|      2016|  429|
|      2017| 1188|
|      2018| 1649|
|      2019| 2016|
|      2020| 1879|
|      2021| 1498|
+----------+-----+



In [24]:
df.createOrReplaceTempView("netflix")

In [25]:
spark.sql("""
SELECT type, COUNT(*) as total
FROM netflix
GROUP BY type
""").show()

+-------+-----+
|   type|total|
+-------+-----+
|TV Show| 2666|
|  Movie| 6131|
+-------+-----+



In [26]:
spark.sql("""
SELECT country, COUNT(*) as total
FROM netflix
GROUP BY country
ORDER BY total DESC
LIMIT 10
""").show()


+--------------+-----+
|       country|total|
+--------------+-----+
| United States| 2812|
|         India|  972|
|       Unknown|  830|
|United Kingdom|  418|
|         Japan|  244|
|   South Korea|  199|
|        Canada|  181|
|         Spain|  145|
|        France|  124|
|        Mexico|  110|
+--------------+-----+



In [29]:
df.write.mode("overwrite").parquet("/content/netflix_parquet")

In [30]:
parquet_df = spark.read.parquet("netflix_parquet")
parquet_df.show(5)

+-------+-----+--------------------+----------------+--------------------+---------------+-------------------+------------+------+--------+--------------------+--------------------+----------+----------------+
|show_id| type|               title|        director|        cast_members|        country|         date_added|release_year|rating|duration|           listed_in|         description|year_added|content_category|
+-------+-----+--------------------+----------------+--------------------+---------------+-------------------+------------+------+--------+--------------------+--------------------+----------+----------------+
|  s1605|Movie|Ari Eldjárn: Pard...|August Jakobsson|         Ari Eldjárn|        Iceland|2020-12-02 00:00:00|        2020| TV-MA|  54 min|     Stand-Up Comedy|In this English-l...|      2020|            Film|
|  s1877|Movie|   Palermo Hollywood|   Eduardo Pinto|Brian Maya, Matía...|      Argentina|2020-10-08 00:00:00|        2004| TV-MA| 106 min|Dramas, Internati...|

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [31]:
!zip -r netflix_parquet.zip /content/netflix_parquet

  adding: content/netflix_parquet/ (stored 0%)
  adding: content/netflix_parquet/part-00001-c0695a4d-2c19-4498-8260-faeb4395faa7-c000.snappy.parquet (deflated 15%)
  adding: content/netflix_parquet/.part-00001-c0695a4d-2c19-4498-8260-faeb4395faa7-c000.snappy.parquet.crc (stored 0%)
  adding: content/netflix_parquet/part-00000-c0695a4d-2c19-4498-8260-faeb4395faa7-c000.snappy.parquet (deflated 15%)
  adding: content/netflix_parquet/.part-00000-c0695a4d-2c19-4498-8260-faeb4395faa7-c000.snappy.parquet.crc (stored 0%)
  adding: content/netflix_parquet/_SUCCESS (stored 0%)
  adding: content/netflix_parquet/._SUCCESS.crc (stored 0%)
